# Blue Catalyst POC — ESM2 Proteome Embeddings (Wetland MUCC vs Rumen PRJEB31266)

This notebook builds a proposal-ready minimal but solid POC:

1. Verify source connectivity (Zenodo MUCC, ENA PRJEB31266).
2. Download a configurable subset of files from each source.
3. Prepare per-genome proteome FASTA inputs for ESM2.
4. Generate mean proteome embeddings per genome using ESM2.
5. Build UMAP/t-SNE visualizations and HDBSCAN clusters.
6. Produce key metrics and a bridging-genome inventory.

> Designed for execution on Apolo-3 (H100) via `jupyter nbconvert --execute` in SLURM.


## Important notes (data reality checks)

- **MUCC v2.0.0**: this notebook queries Zenodo API. If `MUCC_v2.0.0_HQMQ_genes.faa.zip` is not present in the record, provide the direct file URL manually in `CFG['mucc_manual_proteome_url']`.
- **PRJEB31266 (ENA)**: the notebook queries `analysis` records and inspects FTP files. In current API responses, records are commonly `*.fa.gz` assemblies, so this notebook supports a **gene-calling fallback** via `prodigal` when protein FASTA is not directly available.
- For proposal timelines, use a **subset mode** first (e.g., 10–50 genomes/source), then scale.


In [ ]:
from pathlib import Path
import os


def infer_project_root() -> Path:
    env_root = os.getenv("METHANET_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "methanet").exists():
            return p
    return cwd


PROJECT_ROOT = infer_project_root()

CFG = {
    # Output roots
    "out_root": PROJECT_ROOT / "results" / "blue_catalyst_poc",
    "data_root": PROJECT_ROOT / "data" / "blue_catalyst_poc",

    # MUCC
    "mucc_zenodo_record": os.getenv("MUCC_ZENODO_RECORD", "14532347"),
    "mucc_target_key": os.getenv("MUCC_TARGET_KEY", "MUCC_v2.0.0_HQMQ_genes.faa.zip"),
    # optional fallback if record does not expose target key
    "mucc_manual_proteome_url": os.getenv("MUCC_MANUAL_PROTEOME_URL") or None,

    # Rumen ENA
    "rumen_study": os.getenv("RUMEN_STUDY", "PRJEB31266"),

    # Sampling controls
    "subset_mode": os.getenv("BC_SUBSET_MODE", "1") == "1",
    "subset_mucc_genomes": int(os.getenv("BC_SUBSET_MUCC", "20")),
    "subset_rumen_genomes": int(os.getenv("BC_SUBSET_RUMEN", "20")),

    # Embedding
    "esm2_model": os.getenv("BC_ESM2_MODEL", "facebook/esm2_t33_650M_UR50D"),
    "esm2_batch_size": int(os.getenv("BC_ESM2_BATCH", "4")),
    "esm2_max_length": int(os.getenv("BC_ESM2_MAXLEN", "1022")),
    "device": os.getenv("BC_DEVICE", "auto"),

    # Data handling
    "max_proteins_per_genome": int(os.getenv("BC_MAX_PROTEINS", "2000")),
    "min_aa_len": int(os.getenv("BC_MIN_AA_LEN", "30")),
    # enabled by default to handle PRJEB31266 nucleotide-only files if needed
    "allow_gene_calling_fallback": os.getenv("RUMEN_ALLOW_GENE_CALLING", "1") == "1",

    # Analysis
    "umap_n_neighbors": int(os.getenv("BC_UMAP_NEIGHBORS", "20")),
    "umap_min_dist": float(os.getenv("BC_UMAP_MINDIST", "0.15")),
    "tsne_perplexity": int(os.getenv("BC_TSNE_PERPLEXITY", "20")),
    "knn_k_bridge": int(os.getenv("BC_BRIDGE_K", "15")),
}

for p in [CFG["out_root"], CFG["data_root"]]:
    p.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(CFG["data_root"] / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", os.environ["HF_HOME"])
os.environ.setdefault("XDG_CACHE_HOME", str(CFG["data_root"] / ".cache"))

print("Project root:", PROJECT_ROOT)
print("Output root:", CFG["out_root"])
print("Data root:", CFG["data_root"])
print("HF cache:", os.environ["HF_HOME"])
print("Subset mode:", CFG["subset_mode"])
print("Gene-calling fallback enabled:", CFG["allow_gene_calling_fallback"])


In [ ]:
import csv
import gzip
import io
import json
import math
import random
import re
import shutil
import subprocess
import urllib.parse
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import SeqIO
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import HDBSCAN
import umap
import plotly.express as px

from methanet.embedding.esm2 import EmbeddingConfig, ESM2Embedder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
def http_get_json(url: str, timeout: int = 60) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        return json.load(resp)


def stream_download(url: str, out_path: Path, chunk_size: int = 1 << 20) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url, timeout=120) as src, out_path.open('wb') as dst:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            dst.write(chunk)


def is_probably_protein(seq: str) -> bool:
    if not seq:
        return False
    dna_chars = set('ACGTNacgtn')
    aa_chars = set('ACDEFGHIKLMNPQRSTVWYBXZJUO*acdefghiklmnpqrstvwybxzjuo')
    seq_set = set(seq)
    dna_fraction = sum(ch in dna_chars for ch in seq) / max(1, len(seq))
    aa_fraction = sum(ch in aa_chars for ch in seq) / max(1, len(seq))
    # protein-like if mostly AA and not overwhelmingly DNA alphabet
    return aa_fraction > 0.95 and dna_fraction < 0.9


def sanitize_sample_id(x: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(x))


def read_fasta_preview(path: Path, n_records: int = 20):
    cnt = 0
    aa_like = 0
    lengths = []
    open_fn = gzip.open if path.suffix == '.gz' else open
    mode = 'rt'
    with open_fn(path, mode) as handle:
        for rec in SeqIO.parse(handle, 'fasta'):
            s = str(rec.seq)
            lengths.append(len(s))
            if is_probably_protein(s):
                aa_like += 1
            cnt += 1
            if cnt >= n_records:
                break
    return {'checked': cnt, 'aa_like': aa_like, 'mean_len': float(np.mean(lengths)) if lengths else 0.0}


In [ ]:
# --- Minimal source connectivity tests (proposal requirement) ---

# 1) MUCC Zenodo record accessibility
zenodo_url = f"https://zenodo.org/api/records/{CFG['mucc_zenodo_record']}"
mucc_record = http_get_json(zenodo_url)
mucc_files = mucc_record.get('files', [])
print('MUCC record reachable:', bool(mucc_files), '| n_files:', len(mucc_files))
print('MUCC file keys:', [f.get('key') for f in mucc_files])

# 2) ENA PRJEB31266 analysis endpoint accessibility
base = 'https://www.ebi.ac.uk/ena/portal/api/search'
query = f'study_accession="{CFG["rumen_study"]}" AND analysis_type="SEQUENCE_ASSEMBLY"'
params = {
    'result': 'analysis',
    'query': query,
    'fields': 'analysis_accession,scientific_name,submitted_ftp',
    'format': 'tsv',
    'limit': '5',
}
ena_url = base + '?' + urllib.parse.urlencode(params)
with urllib.request.urlopen(ena_url, timeout=60) as resp:
    ena_tsv = resp.read().decode('utf-8', errors='ignore')
rows = [r for r in ena_tsv.splitlines() if r.strip()]
print('ENA query reachable:', len(rows) > 1, '| rows_returned:', max(0, len(rows) - 1))
print('ENA header:', rows[0] if rows else 'NA')
print('ENA first row:', rows[1] if len(rows) > 1 else 'NA')


In [ ]:
# --- MUCC download logic ---

mucc_dir = CFG["data_root"] / "mucc"
mucc_dir.mkdir(parents=True, exist_ok=True)

mucc_file_map = {f["key"]: f for f in mucc_files}
mucc_target = CFG["mucc_target_key"]
mucc_download_path = None

if mucc_target in mucc_file_map:
    entry = mucc_file_map[mucc_target]
    mucc_download_path = mucc_dir / mucc_target
    if not mucc_download_path.exists():
        print("Downloading MUCC proteome bundle:", mucc_target)
        stream_download(entry["links"]["self"], mucc_download_path)
    else:
        print("MUCC proteome bundle already exists:", mucc_download_path)
elif CFG["mucc_manual_proteome_url"]:
    mucc_download_path = mucc_dir / Path(
        urllib.parse.urlparse(CFG["mucc_manual_proteome_url"]).path
    ).name
    if not mucc_download_path.exists():
        print("Downloading MUCC proteome bundle from manual URL...")
        stream_download(CFG["mucc_manual_proteome_url"], mucc_download_path)
    else:
        print("MUCC manual proteome bundle already exists:", mucc_download_path)
else:
    # Fallback for the current public record content
    alt_key = "Methanoregula_MAGs_DB.zip"
    if alt_key in mucc_file_map:
        entry = mucc_file_map[alt_key]
        mucc_download_path = mucc_dir / alt_key
        if not mucc_download_path.exists():
            print("Downloading MUCC fallback MAG archive:", alt_key)
            stream_download(entry["links"]["self"], mucc_download_path)
        else:
            print("MUCC fallback MAG archive already exists:", mucc_download_path)
    else:
        print(
            "WARNING: target MUCC proteome key not found and no fallback/manual URL provided."
        )
        print("Set CFG['mucc_manual_proteome_url'] and rerun this cell.")

print("MUCC download path:", mucc_download_path)


In [ ]:
# --- Rumen (PRJEB31266) manifest + subset downloads ---

rumen_dir = CFG['data_root'] / 'rumen'
rumen_dir.mkdir(parents=True, exist_ok=True)

params = {
    'result': 'analysis',
    'query': query,
    'fields': 'analysis_accession,scientific_name,analysis_alias,submitted_ftp',
    'format': 'tsv',
    'limit': '0',
}
manifest_url = base + '?' + urllib.parse.urlencode(params)
with urllib.request.urlopen(manifest_url, timeout=180) as resp:
    tsv = resp.read().decode('utf-8', errors='ignore')

rumen_rows = []
reader = csv.DictReader(io.StringIO(tsv), delimiter='	')
for row in reader:
    ftp_field = (row.get('submitted_ftp') or '').strip()
    if not ftp_field:
        continue
    for rel in ftp_field.split(';'):
        rel = rel.strip()
        if not rel:
            continue
        rumen_rows.append({
            'analysis_accession': row.get('analysis_accession', ''),
            'scientific_name': row.get('scientific_name', ''),
            'analysis_alias': row.get('analysis_alias', ''),
            'submitted_ftp_rel': rel,
            'download_url': 'https://' + rel,
            'filename': Path(rel).name,
        })

rumen_manifest = pd.DataFrame(rumen_rows).drop_duplicates(subset=['download_url'])
rumen_manifest['ecosystem'] = 'rumen'
rumen_manifest['domain'] = np.where(
    rumen_manifest['scientific_name'].str.contains('archae|euryarchae', case=False, na=False),
    'Archaea',
    'Bacteria',
)

manifest_path = rumen_dir / 'prjeb31266_analysis_manifest.tsv'
rumen_manifest.to_csv(manifest_path, sep='	', index=False)
print('Rumen manifest rows:', len(rumen_manifest), '| saved to', manifest_path)

# subset download
subset_n = CFG['subset_rumen_genomes'] if CFG['subset_mode'] else len(rumen_manifest)
rumen_subset = rumen_manifest.head(subset_n).copy()

for _, r in rumen_subset.iterrows():
    out_fp = rumen_dir / 'raw' / r['filename']
    if out_fp.exists():
        continue
    stream_download(r['download_url'], out_fp)

print('Downloaded rumen subset files:', subset_n)

# quick content test on first file
first_file = rumen_dir / 'raw' / rumen_subset.iloc[0]['filename']
preview = read_fasta_preview(first_file, n_records=20)
print('First rumen file preview:', first_file.name, preview)


In [ ]:
# --- Build per-genome proteome FASTA directory for ESM2 ---

proteome_dir = CFG['data_root'] / 'proteomes'
proteome_dir.mkdir(parents=True, exist_ok=True)

sample_records = []


def ensure_prodigal_available():
    if shutil.which('prodigal') is None:
        raise RuntimeError(
            'Gene-calling fallback is enabled, but prodigal is not available on PATH.'
        )


def maybe_decompress_to_temp(src_path: Path, tmp_dir: Path) -> Path:
    if src_path.suffix != '.gz':
        return src_path
    tmp_dir.mkdir(parents=True, exist_ok=True)
    out_path = tmp_dir / src_path.with_suffix('').name
    if out_path.exists():
        return out_path

    try:
        with gzip.open(src_path, 'rb') as src, out_path.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    except (EOFError, OSError) as e:
        if out_path.exists():
            out_path.unlink(missing_ok=True)
        raise RuntimeError(f'Corrupted or incomplete gzip file: {src_path}') from e

    return out_path


def select_zip_members(zip_path: Path, subset_n: int | None):
    import zipfile

    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = [n for n in zf.namelist() if not n.endswith('/')]

    protein_exts = ('.faa', '.faa.gz', '.fa', '.fa.gz')
    nucleotide_exts = ('.fna', '.fna.gz', '.fasta', '.fasta.gz')

    prot = [n for n in names if n.lower().endswith(protein_exts)]
    nuc = [n for n in names if n.lower().endswith(nucleotide_exts)]

    if prot:
        chosen = prot
        is_protein = True
    else:
        chosen = nuc
        is_protein = False

    chosen = sorted(chosen)
    if subset_n is not None:
        chosen = chosen[:subset_n]

    return chosen, is_protein


# A) MUCC
if mucc_download_path and mucc_download_path.exists():
    mucc_extract_dir = mucc_dir / 'extracted'
    mucc_extract_dir.mkdir(exist_ok=True, parents=True)

    subset_n = CFG['subset_mucc_genomes'] if CFG['subset_mode'] else None

    extracted_paths = []
    if str(mucc_download_path).endswith('.zip'):
        import zipfile

        chosen_members, zip_is_protein = select_zip_members(mucc_download_path, subset_n)
        if not chosen_members:
            print('WARNING: No suitable FASTA members found in MUCC zip.')
        else:
            print('MUCC zip members selected:', len(chosen_members), '| protein_mode:', zip_is_protein)
            with zipfile.ZipFile(mucc_download_path, 'r') as zf:
                for member in chosen_members:
                    target_path = mucc_extract_dir / member
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    if not target_path.exists():
                        with zf.open(member) as src, target_path.open('wb') as dst:
                            shutil.copyfileobj(src, dst)
                    extracted_paths.append(target_path)

            if zip_is_protein:
                for fp in extracted_paths:
                    sample = sanitize_sample_id(fp.stem.replace('.faa', '').replace('.fa', ''))
                    out_fp = proteome_dir / f'mucc__{sample}.faa'
                    try:
                        if fp.suffix == '.gz':
                            with gzip.open(fp, 'rb') as src, out_fp.open('wb') as dst:
                                shutil.copyfileobj(src, dst)
                        else:
                            shutil.copyfile(fp, out_fp)
                    except (EOFError, OSError) as e:
                        print(f'SKIP MUCC file due to gzip/read error: {fp} ({e})')
                        continue
                    sample_records.append(
                        {
                            'sample': out_fp.stem,
                            'source': 'mucc',
                            'ecosystem': 'wetland',
                            'domain': 'Unknown',
                            'proteome_faa': str(out_fp),
                        }
                    )
            elif CFG['allow_gene_calling_fallback']:
                ensure_prodigal_available()
                tmp_mucc_nuc_dir = proteome_dir / '_tmp_mucc_nuc'
                for fp in extracted_paths:
                    sample = sanitize_sample_id(
                        fp.stem.replace('.fna', '').replace('.fasta', '').replace('.fa', '')
                    )
                    out_fp = proteome_dir / f'mucc__{sample}.faa'
                    try:
                        nuc_fp = maybe_decompress_to_temp(fp, tmp_mucc_nuc_dir)
                    except RuntimeError as e:
                        print(f'SKIP MUCC nucleotide file due to gzip/read error: {fp} ({e})')
                        continue
                    cmd = ['prodigal', '-i', str(nuc_fp), '-a', str(out_fp), '-p', 'meta', '-q']
                    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    sample_records.append(
                        {
                            'sample': out_fp.stem,
                            'source': 'mucc',
                            'ecosystem': 'wetland',
                            'domain': 'Unknown',
                            'proteome_faa': str(out_fp),
                        }
                    )
                print('MUCC fallback via prodigal completed for', len(extracted_paths), 'genomes.')
            else:
                print('WARNING: MUCC zip appears nucleotide-only and fallback is disabled.')

    elif str(mucc_download_path).endswith('.gz'):
        out_file = mucc_extract_dir / mucc_download_path.with_suffix('').name
        if not out_file.exists():
            with gzip.open(mucc_download_path, 'rb') as src, out_file.open('wb') as dst:
                shutil.copyfileobj(src, dst)

        faa_candidates = [out_file] if out_file.suffix in {'.faa', '.fa'} else []
        if faa_candidates:
            for fp in faa_candidates:
                sample = sanitize_sample_id(fp.stem.replace('.faa', '').replace('.fa', ''))
                out_fp = proteome_dir / f'mucc__{sample}.faa'
                shutil.copyfile(fp, out_fp)
                sample_records.append(
                    {
                        'sample': out_fp.stem,
                        'source': 'mucc',
                        'ecosystem': 'wetland',
                        'domain': 'Unknown',
                        'proteome_faa': str(out_fp),
                    }
                )

# B) Rumen
rumen_raw_dir = rumen_dir / 'raw'
if rumen_raw_dir.exists():
    rumen_files = sorted(rumen_raw_dir.glob('*.fa.gz'))
    if CFG['subset_mode']:
        rumen_files = rumen_files[: CFG['subset_rumen_genomes']]

    rumen_domain_lookup = {}
    if 'rumen_manifest' in globals() and not rumen_manifest.empty:
        tmp = rumen_manifest[['filename', 'domain']].drop_duplicates()
        rumen_domain_lookup = dict(zip(tmp['filename'], tmp['domain']))

    tmp_rumen_nuc_dir = proteome_dir / '_tmp_rumen_nuc'

    for fp in rumen_files:
        try:
            prev = read_fasta_preview(fp, n_records=20)
        except (EOFError, OSError) as e:
            print(f'SKIP RUMEN file due to gzip/read error during preview: {fp} ({e})')
            continue

        sample = sanitize_sample_id(fp.stem.replace('.fa', ''))
        out_fp = proteome_dir / f'rumen__{sample}.faa'

        if prev['aa_like'] >= max(1, int(0.7 * prev['checked'])):
            try:
                with gzip.open(fp, 'rb') as src, out_fp.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
            except (EOFError, OSError) as e:
                print(f'SKIP RUMEN protein-like file due to gzip/read error: {fp} ({e})')
                continue
            domain = rumen_domain_lookup.get(fp.name, 'Bacteria')
            sample_records.append(
                {
                    'sample': out_fp.stem,
                    'source': 'rumen',
                    'ecosystem': 'rumen',
                    'domain': domain,
                    'proteome_faa': str(out_fp),
                }
            )
        elif CFG['allow_gene_calling_fallback']:
            ensure_prodigal_available()
            try:
                nuc_fp = maybe_decompress_to_temp(fp, tmp_rumen_nuc_dir)
            except RuntimeError as e:
                print(f'SKIP RUMEN nucleotide file due to gzip/read error: {fp} ({e})')
                continue
            cmd = ['prodigal', '-i', str(nuc_fp), '-a', str(out_fp), '-p', 'meta', '-q']
            subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            domain = rumen_domain_lookup.get(fp.name, 'Bacteria')
            sample_records.append(
                {
                    'sample': out_fp.stem,
                    'source': 'rumen',
                    'ecosystem': 'rumen',
                    'domain': domain,
                    'proteome_faa': str(out_fp),
                }
            )
        else:
            print(f'SKIP (non-protein detected, fallback disabled): {fp} -> {prev}')

sample_df = pd.DataFrame(sample_records)
sample_manifest = CFG['out_root'] / 'proteome_sample_manifest.tsv'
sample_df.to_csv(sample_manifest, sep='\t', index=False)

print('Prepared proteome samples:', len(sample_df))
print(sample_df.head())
print('Manifest:', sample_manifest)

In [ ]:
# --- ESM2 embedding: protein -> genome mean embedding ---

sample_df = pd.read_csv(CFG['out_root'] / 'proteome_sample_manifest.tsv', sep='	')
if sample_df.empty:
    raise RuntimeError('No proteome samples prepared. Check previous cells and source availability.')

if CFG['subset_mode']:
    # keep manageable runtime for proposal sprint
    cap = CFG['subset_mucc_genomes'] + CFG['subset_rumen_genomes']
    sample_df = sample_df.head(cap).copy()

emb_cfg = EmbeddingConfig(
    model_name=CFG['esm2_model'],
    batch_size=CFG['esm2_batch_size'],
    max_length=CFG['esm2_max_length'],
    device=CFG['device'],
)
embedder = ESM2Embedder(emb_cfg)

embeddings = []
kept_rows = []

for row in sample_df.itertuples(index=False):
    fp = Path(row.proteome_faa)
    seqs = []
    ids = []
    with fp.open() as h:
        for rec in SeqIO.parse(h, 'fasta'):
            seq = str(rec.seq)
            if len(seq) < CFG['min_aa_len']:
                continue
            seqs.append(seq)
            ids.append(rec.id)
            if len(seqs) >= CFG['max_proteins_per_genome']:
                break

    if not seqs:
        print('No valid proteins:', fp.name)
        continue

    prot_emb = embedder.embed_proteins(seqs, ids)
    genome_emb = embedder.embed_genome(prot_emb, aggregation='mean')

    embeddings.append(genome_emb.astype(np.float32))
    kept_rows.append(dict(row._asdict(), n_proteins_used=len(seqs)))

emb_mat = np.vstack(embeddings)
meta_df = pd.DataFrame(kept_rows)

out_npz = CFG['out_root'] / 'genome_embeddings.npz'
np.savez_compressed(
    out_npz,
    embeddings=emb_mat,
    sample=meta_df['sample'].values,
    source=meta_df['source'].values,
    ecosystem=meta_df['ecosystem'].values,
    domain=meta_df['domain'].values,
    n_proteins_used=meta_df['n_proteins_used'].values,
)

meta_df.to_csv(CFG['out_root'] / 'embedding_metadata.tsv', sep='	', index=False)
print('Embedding matrix:', emb_mat.shape, '| saved:', out_npz)


In [ ]:
# --- Dimensionality reduction + clustering + metrics ---

bundle = np.load(CFG['out_root'] / 'genome_embeddings.npz', allow_pickle=True)
X = bundle['embeddings']
meta = pd.DataFrame({
    'sample': bundle['sample'].astype(str),
    'source': bundle['source'].astype(str),
    'ecosystem': bundle['ecosystem'].astype(str),
    'domain': bundle['domain'].astype(str),
    'n_proteins_used': bundle['n_proteins_used'],
})

# basic finite filter
mask = np.isfinite(X).all(axis=1)
X = X[mask]
meta = meta.loc[mask].reset_index(drop=True)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=CFG['umap_n_neighbors'],
    min_dist=CFG['umap_min_dist'],
    metric='cosine',
    random_state=SEED,
)
U = umap_model.fit_transform(X)

# t-SNE
tsne_model = TSNE(
    n_components=2,
    perplexity=min(CFG['tsne_perplexity'], max(5, (len(X)-1)//3)),
    random_state=SEED,
    init='pca',
    learning_rate='auto',
)
T = tsne_model.fit_transform(X)

# HDBSCAN
hdb = HDBSCAN(min_cluster_size=max(4, len(X)//20), metric='euclidean')
cluster_labels = hdb.fit_predict(X)
meta['cluster'] = cluster_labels
meta['umap_1'] = U[:, 0]
meta['umap_2'] = U[:, 1]
meta['tsne_1'] = T[:, 0]
meta['tsne_2'] = T[:, 1]

# Metrics
metrics = {}
non_noise = cluster_labels >= 0
if non_noise.sum() > 2 and len(set(cluster_labels[non_noise])) > 1:
    metrics['silhouette_non_noise'] = float(silhouette_score(X[non_noise], cluster_labels[non_noise], metric='euclidean'))
else:
    metrics['silhouette_non_noise'] = float('nan')

# Purity by ecosystem/domain
def purity(df, label_col):
    vals = []
    for c, sub in df[df['cluster'] >= 0].groupby('cluster'):
        counts = sub[label_col].value_counts()
        vals.append(counts.iloc[0] / counts.sum())
    return float(np.mean(vals)) if vals else float('nan')

metrics['cluster_purity_ecosystem'] = purity(meta, 'ecosystem')
metrics['cluster_purity_domain'] = purity(meta, 'domain')
metrics['n_clusters_excluding_noise'] = int(len(set(cluster_labels[cluster_labels >= 0])))
metrics['noise_fraction'] = float((cluster_labels < 0).mean())

# Bridging score from ecosystem entropy in local neighborhood
nbrs = NearestNeighbors(n_neighbors=min(CFG['knn_k_bridge'], len(X))).fit(X)
idx = nbrs.kneighbors(return_distance=False)

bridging_scores = []
for i, nn in enumerate(idx):
    eco = meta.loc[nn, 'ecosystem'].values
    cnt = Counter(eco)
    probs = np.array([v / len(eco) for v in cnt.values()], dtype=float)
    entropy = -np.sum(probs * np.log2(probs + 1e-12))
    bridging_scores.append(float(entropy))

meta['bridging_score'] = bridging_scores
bridge_df = meta.sort_values('bridging_score', ascending=False).head(min(50, len(meta))).copy()

# Save artifacts
meta.to_csv(CFG['out_root'] / 'embedding_projection_clusters.tsv', sep='	', index=False)
bridge_df.to_csv(CFG['out_root'] / 'bridging_genomes_top.tsv', sep='	', index=False)
with (CFG['out_root'] / 'poc_metrics.json').open('w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics:', json.dumps(metrics, indent=2))
print('Saved:', CFG['out_root'] / 'embedding_projection_clusters.tsv')
print('Saved:', CFG['out_root'] / 'bridging_genomes_top.tsv')


In [ ]:
# --- Proposal-ready visualizations (interactive HTML) ---

proj = pd.read_csv(CFG['out_root'] / 'embedding_projection_clusters.tsv', sep='	')

fig_umap_eco = px.scatter(
    proj, x='umap_1', y='umap_2', color='ecosystem', symbol='domain',
    hover_data=['sample', 'cluster', 'bridging_score', 'n_proteins_used'],
    title='UMAP — ESM2 proteome embeddings by ecosystem/domain',
    width=1000, height=700,
)

fig_umap_cluster = px.scatter(
    proj, x='umap_1', y='umap_2', color=proj['cluster'].astype(str), symbol='ecosystem',
    hover_data=['sample', 'domain', 'bridging_score'],
    title='UMAP — HDBSCAN cluster map',
    width=1000, height=700,
)

fig_tsne = px.scatter(
    proj, x='tsne_1', y='tsne_2', color='ecosystem', symbol='domain',
    hover_data=['sample', 'cluster', 'bridging_score'],
    title='t-SNE — ESM2 proteome embeddings',
    width=1000, height=700,
)

fig_umap_eco.write_html(CFG['out_root'] / 'umap_ecosystem_domain.html')
fig_umap_cluster.write_html(CFG['out_root'] / 'umap_hdbscan_clusters.html')
fig_tsne.write_html(CFG['out_root'] / 'tsne_ecosystem_domain.html')

print('Saved HTML figures to:', CFG['out_root'])
fig_umap_eco.show()


## Final deliverables generated by this notebook

- `results/blue_catalyst_poc/genome_embeddings.npz`
- `results/blue_catalyst_poc/embedding_metadata.tsv`
- `results/blue_catalyst_poc/embedding_projection_clusters.tsv`
- `results/blue_catalyst_poc/bridging_genomes_top.tsv`
- `results/blue_catalyst_poc/poc_metrics.json`
- `results/blue_catalyst_poc/umap_ecosystem_domain.html`
- `results/blue_catalyst_poc/umap_hdbscan_clusters.html`
- `results/blue_catalyst_poc/tsne_ecosystem_domain.html`

These are the core artifacts for Blue Catalyst proposal visuals and metric tables.
